# ECG-FM Stage 2 — Kaggle One-Click

### Before running:

**1. Add datasets** (Edit → Settings → Add-ons):
- `shlomihazan/ptbxl-500hz` — PTB-XL 500 Hz signals
- `shlomihazan/ecgfm-models` — model weights + training scripts

**2. Set GPU to T4:** Edit → Settings → Accelerator → **GPU T4 x2**
- NOTE: P100 is NOT compatible with the current PyTorch — T4 is required.

**3. Run the single cell below.**

---
Output `ecgfm_stage2.pt` will appear in the Output tab when done (~3 hrs).

In [ ]:
import os, shutil, zipfile, torch, psutil
from pathlib import Path

# ── 1. Check GPU — must be T4 (sm_75+), not P100 (sm_60) ─────────────────────
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Edit → Settings → Accelerator → GPU T4 x2')

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU : {gpu_name}  (sm_{major}{minor})')
print(f'VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB')
print(f'RAM : {round(psutil.virtual_memory().total / 1e9, 1)} GB')

if major < 7:
    raise RuntimeError(
        f'{gpu_name} (sm_{major}{minor}) is NOT compatible with PyTorch 2.x.\n'
        f'Please go to Edit → Settings → Accelerator and switch to GPU T4 x2 (sm_75).'
    )
print('GPU OK\n')

# ── 2. Auto-detect input datasets (handles any mount path) ───────────────────
print('Scanning /kaggle/input ...')
for p in sorted(Path('/kaggle/input').rglob('*')):
    if p.is_file():
        print(f'  {p}')

# Find PTB-XL: look for ptbxl_database.csv anywhere under /kaggle/input
PTBXL_PATH = None
for f in Path('/kaggle/input').rglob('ptbxl_database.csv'):
    PTBXL_PATH = str(f.parent)
    break

# Find ecgfm-models: look for ecgfm_stage1.pt anywhere under /kaggle/input
MODEL_PATH = None
for f in Path('/kaggle/input').rglob('ecgfm_stage1.pt'):
    MODEL_PATH = str(f.parent)
    break

if not PTBXL_PATH:
    raise FileNotFoundError('PTB-XL not found — add shlomihazan/ptbxl-500hz in Edit → Settings → Add-ons.')
if not MODEL_PATH:
    raise FileNotFoundError('ecgfm-models not found — add shlomihazan/ecgfm-models in Edit → Settings → Add-ons.')

for fname in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py',
              'mimic_iv_ecg_physionet_pretrained.pt']:
    if not os.path.exists(f'{MODEL_PATH}/{fname}'):
        raise FileNotFoundError(f'Missing {fname} in ecgfm-models dataset (needs v2+).')

print(f'\nPTB-XL path : {PTBXL_PATH}')
print(f'Model path  : {MODEL_PATH}\n')

# ── 3. Set up working directory ───────────────────────────────────────────────
WORKDIR = '/kaggle/working/ecgfm'
os.makedirs(f'{WORKDIR}/models/ecgfm', exist_ok=True)
os.makedirs(f'{WORKDIR}/ekg_datasets', exist_ok=True)

for script in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py']:
    shutil.copy(f'{MODEL_PATH}/{script}', f'{WORKDIR}/{script}')

shutil.copy(f'{MODEL_PATH}/mimic_iv_ecg_physionet_pretrained.pt',
            f'{WORKDIR}/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_PATH}/ecgfm_stage1.pt', f'{WORKDIR}/models/ecgfm_stage1.pt')

# Extract or symlink PTB-XL
ptbxl_dest = f'{WORKDIR}/ekg_datasets/ptbxl'
if os.path.islink(ptbxl_dest):
    os.remove(ptbxl_dest)

zip_path = f'{PTBXL_PATH}/records500.zip'
if os.path.exists(zip_path):
    os.makedirs(ptbxl_dest, exist_ok=True)
    print('Extracting records500.zip (~2 min)...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(ptbxl_dest)
    shutil.copy(f'{PTBXL_PATH}/ptbxl_database.csv', f'{ptbxl_dest}/ptbxl_database.csv')
    shutil.copy(f'{PTBXL_PATH}/scp_statements.csv',  f'{ptbxl_dest}/scp_statements.csv')
else:
    os.symlink(os.path.abspath(PTBXL_PATH), ptbxl_dest)

n_dat = len(list(Path(ptbxl_dest).rglob('*.dat')))
print(f'PTB-XL signal files: {n_dat}/21837')
if n_dat < 20000:
    raise RuntimeError(f'Only {n_dat} signal files — dataset may be incomplete.')

print(f'Encoder : {round(os.path.getsize(WORKDIR+"/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt")/1e6)} MB')
print(f'Stage1  : {round(os.path.getsize(WORKDIR+"/models/ecgfm_stage1.pt")/1e6)} MB')
print('Setup complete\n')

# ── 4. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb', 'scipy', 'scikit-learn'],
               check=True)
print('Dependencies ready\n')

# ── 5. Train ──────────────────────────────────────────────────────────────────
# ~20 min/epoch on T4, ~3 hrs total. Saves models/ecgfm_stage2.pt each improvement.
# OOM? change --batch_s2 8 to --batch_s2 4
os.chdir(WORKDIR)
!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

# ── 6. Results ────────────────────────────────────────────────────────────────
model_path = f'{WORKDIR}/models/ecgfm_stage2.pt'
if os.path.exists(model_path):
    print(f'\necgfm_stage2.pt saved ({round(os.path.getsize(model_path)/1e6)} MB)')
    print('Download: Output tab → ecgfm/models/ecgfm_stage2.pt\n')
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print('Final test results:')
    print(f"  Accuracy : {tm.get('acc',  0):.1%}")
    print(f"  HYP F1   : {tm.get('hyp_f1',   0):.3f}  (baseline 0.375)")
    print(f"  Macro F1 : {tm.get('macro_f1', 0):.3f}  (baseline 0.492)")
else:
    print('WARNING: ecgfm_stage2.pt not found — check output above for errors')